# Homework 4: Model Evaluation and Comparison

BEE 4850/5850, Fall 2025

**Name**:

**ID**:

> **Due Date**
>
> Friday, 3/27/26, 9:00pm

## Overview

### Instructions

The goal of this homework assignment is to practice model evaluation and
using cross-validation and information criteria to distinguish between
models.

-   Problem 1 asks you to fit a model for cherry blossom bloom dates and
    use cross-validation to draw conclusions about generalizability.
-   Problem 2 asks you to compare several linear regression models for
    environmental influences on non-accidental death data.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/Documents/Spring_2026/BEE4850/hw/hw04-avinasharavind`
Precompiling project...
   1524.8 ms  ✓ EnumX
   1259.1 ms  ✓ Adapt
   1494.1 ms  ✓ EpollShim_jll
   1496.9 ms  ✓ Libmount_jll
    542.7 ms  ✓ Xorg_libICE_jll
    554.0 ms  ✓ Xorg_libpciaccess_jll
    548.8 ms  ✓ Xorg_libXau_jll
   1906.6 ms  ✓ MbedTLS
    595.6 ms  ✓ LERC_jll
    567.6 ms  ✓ mtdev_jll
    565.6 ms  ✓ Xorg_libXdmcp_jll
    704.7 ms  ✓ JpegTurbo_jll
    718.4 ms  ✓ XZ_jll
    775.5 ms  ✓ Arpack_jll
    702.7 ms  ✓ Xorg_xtrans_jll
    710.2 ms  ✓ libevdev_jll
    774.4 ms  ✓ LZO_jll
    816.5 ms  ✓ Zstd_jll
    787.4 ms  ✓ eudev_jll
    751.1 ms  ✓ Dbus_jll
    983.7 ms  ✓ ColorVectorSpace → SpecialFunctionsExt
   2908.1 ms  ✓ SpecialFunctions → SpecialFunctionsChainRulesCoreExt
   2241.4 ms  ✓ StatsFuns → StatsFunsChainRulesCoreExt
   3151.6 ms  ✓ OpenSSL
   2446.8 ms  ✓ Distributions → DistributionsTestExt
    879.3 ms  ✓ ArrayInterface
    733.8 ms  ✓ OffsetArrays → OffsetArraysAdaptExt
 

The following packages are included in the environment (to help you find
other similar packages in other languages). The code below loads these
packages for use in the subsequent notebook (the desired functionality
for each package is commented next to the package).

In [2]:
using Random # random number generation and seed-setting
using DataFrames # tabular data structure
using DataFramesMeta # API which can simplify chains of DataFrames transformations
using CSV # reads/writes .csv files
using Distributions # interface to work with probability distributions
using Plots # plotting library
using StatsBase # statistical quantities like mean, median, etc
using StatsPlots # some additional statistical plotting tools
using Optim # optimization package for model fitting
using Dates # API for date-time data structures

In [51]:
blossoms = CSV.read("data/kyoto.csv", DataFrame, type=Float32, missingstring="NA")

┌ Warning: `type` keyword argument is deprecated; a single type can be passed to `types` instead
└ @ CSV /Users/avinasharavind/.julia/packages/CSV/LiiJM/src/context.jl:356


Row,Year.AD,Flowering.DOY,Flowering.MMDD
,Float32,Float32?,Float32?
1,801.0,missing,missing
2,802.0,missing,missing
3,803.0,missing,missing
4,804.0,missing,missing
5,805.0,missing,missing
6,806.0,missing,missing
7,807.0,missing,missing
8,808.0,missing,missing
9,809.0,missing,missing


In [68]:
temp_dat = CSV.read("data/HadCRUT5.1Analysis_gl.txt", delim=" ", 
    ignorerepeated=true,
    header=["Year", "January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December", "Annual"],
    silencewarnings=true, DataFrame) 
temp_dat = temp_dat[3:2:nrow(temp_dat), :]
start = findfirst(==(1851.), temp_dat[!, "Year"])
finish = findfirst(==(2015.), temp_dat[!, "Year"])

temp_dat = temp_dat[start:finish, :]

Row,Year,January,February,March,April,May,June,July,August,September,October,November,December,Annual
,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64?
1,1851,-0.266,-0.467,-0.641,-0.528,-0.222,-0.207,-0.136,-0.148,-0.146,-0.045,-0.108,-0.248,-0.264
2,1852,-0.346,-0.496,-0.567,-0.575,-0.113,-0.077,0.009,-0.149,-0.006,-0.158,-0.275,0.064,-0.224
3,1853,-0.204,-0.399,-0.273,-0.401,-0.27,-0.124,-0.072,-0.04,-0.266,-0.373,-0.401,-0.326,-0.262
4,1854,-0.403,-0.392,-0.247,-0.327,-0.277,-0.302,-0.162,-0.233,-0.216,-0.099,-0.412,-0.454,-0.294
5,1855,-0.17,-0.313,-0.345,-0.211,-0.368,-0.333,-0.315,-0.202,-0.259,-0.2,-0.374,-0.408,-0.291
6,1856,-0.083,-0.347,-0.371,-0.314,-0.293,-0.225,-0.223,-0.249,-0.402,-0.446,-0.561,-0.423,-0.328
7,1857,-0.494,-0.442,-0.488,-0.675,-0.654,-0.291,-0.404,-0.356,-0.368,-0.467,-0.662,-0.289,-0.466
8,1858,-0.336,-0.488,-0.566,-0.43,-0.523,-0.36,-0.341,-0.288,-0.28,-0.292,-0.404,-0.455,-0.397
9,1859,-0.405,-0.351,-0.322,-0.189,-0.17,-0.24,-0.29,-0.266,-0.401,-0.247,-0.31,-0.311,-0.292


In [73]:
#MLE Log-Likelihood Estimationm
function  linear_loglik(theta, blossoms, march)
    beta0, beta1, sigma = theta
    mean = beta0 .+ (beta1*march)
    ll = sum(logpdf.(skipmissing.(Normal.(mean, sigma), blossoms)))
    return ll
end

lb = [0.0, -1000.0, 1.0]
ub = [1000.0, 1000.0, 100.0]
theta0 = [500.0, 0.0, 50.0]
optim_out = Optim.optimize(theta -> -linear_loglik(theta, blossoms[!, " Flowering.DOY"], temp_dat[!, "March"]), lb, ub, theta0)
theta_mle = optim_out.minimizer
@show theta_mle

MethodError: MethodError: no method matching skipmissing(::Normal{Float64}, ::Float32)
The function `skipmissing` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  skipmissing(::Any)
   @ Base missing.jl:232


#### Problem 1.2

Calculate the log-loss (negative log-probability) and mean-squared error
metrics for the fitted model on the in-sample data.

#### Problem 1.3

Conduct a 5-fold cross-validation for the model using both of these
metrics. How do they compare to the in-sample metrics?

#### Problem 1.4

What conclusions can you draw about the model’s generalizability?

### Problem 2

Let’s re-examine the influence of air pollution and temperature on
mortality in Chicago, IL. The `data/chicago.csv` dataset (originally
from the [`gamair` `R`
package](https://cran.r-project.org/web/packages/gamair/index.html))contains
data on the relationship between environmental conditions (temperature
and air quality) and deaths in Chicago from 1987–2000. The variables
are:

-   the total number of (non-accidental) deaths each day (`death`);
-   the median density over the city of large pollutant particles (PM10;
    `pm10median`);
-   the median density of smaller pollutant particles (PM2.5;
    `pm25median`);
-   the median concentration of ozone (O$_3$) in the air (`o3median`);
    the median concentration of sulfur dioxide (SO$_2$) in the air
    (`so2median`);
-   the time in days (`time`); and the daily mean temperature (`tmpd`).

The pollution variables have been standardized to represent deviations
from the overall median and the temperature is in Fahrenheit. The days
are given as a number of days relative to December 31, 1993. Note that
some values are missing; these can be treated as missing at random and
dropped in your regressions.

In [ ]:
chicago_dat = CSV.read("data/chicago.csv", DataFrame; delim=',', header=true, missingstring="NA")
day_zero = Date("1993-12-31")
chicago_dat.Date = day_zero .+ Day.(chicago_dat.time .+ 0.5)

plot(chicago_dat.Date, chicago_dat.death, lw=2, xlabel="Date", ylabel="Non-Accidental Deaths)", legend=false)

#### Problem 2.1

Load and plot the deaths data. Linearly regress the number of deaths
against time and report the coefficients for the fitted model. Add the
regression line to your plot. What does this regression tell you?

#### Problem 2.2

Now linearly regress the deaths data against temperature. Once again,
report the coefficients, and add the regression line to your plot. What
does this regression tell you about the influence of temperature?

#### Problem 2.3

Expand your previous model by regressing deaths against temperature and
PM2.5 density. Report the coefficients and add this regression line to
your plot.

#### Problem 2.4

Conduct a 5-fold cross-validation for the temperature only and
temperature and PM2.5 models, using both the mean-squared-error and
log-probability metrics. Are there any structures in the dataset that
you need to account for when developing your cross-validation folds?
Which model performs the best? What can you conclude from this analysis?

#### Problem 2.5

Compute the AICs for the three fitted models and compare them. What do
they tell you about the relative evidence for the influence of
temperature and PM2.5 density on deaths? How do your conclusions based
on AIC compare to those you might draw from your regression
coefficients, plots, and cross-validation?

## References